# 08 Isolation Forest, confidence intervals and rule check
Run after notebooks 01 to 06. Writes to the results folder set in config.py.

## Part A. Isolation Forest (two variants), fitted per building on training rows only, no labels

In [ ]:
import sys, json, time
from config import *
from sklearn.ensemble import IsolationForest

pred = pd.read_csv(OUT_DIR/"predictions.csv.gz", parse_dates=["timestamp"])
lab = pd.read_csv(OUT_DIR/"lead_labels.csv.gz", parse_dates=["timestamp"])[["building_id","timestamp","label","in_zero_run"]]
d = pred.merge(lab, on=["building_id","timestamp"], how="left")
d["in_zero_run"] = d["in_zero_run"].fillna(False).astype(bool)
d = d.sort_values(["building_id","timestamp"]).reset_index(drop=True)
d["y"] = d["meter_reading"].where(~d["in_zero_run"])
d["ly"] = np.log1p(d["y"])
g = d.groupby("building_id")["ly"]
d["e_C"] = d["ly"] - np.log1p(d["pred_C"])
d["d1"] = d["ly"] - g.shift(1)
d["d24"] = d["ly"] - g.shift(24)
d["hour"] = d["timestamp"].dt.hour
d["wk"] = (d["timestamp"].dt.dayofweek >= 5).astype(int)
VARIANTS = {"iforest_C": ["e_C","hour","wk"], "iforest_raw": ["ly","d1","d24","hour","wk"]}
for v in VARIANTS: d["score_"+v] = np.nan
t0=time.time()
for b, idx in d.groupby("building_id").groups.items():
    s = d.loc[idx]
    for v, f in VARIANTS.items():
        ok = s[f].notna().all(axis=1)
        tr = s[ok & (s["timestamp"] < TRAIN_END)]
        if len(tr) < 200: continue
        m = IsolationForest(n_estimators=100, max_samples=256, random_state=0).fit(tr[f])
        te = s[ok]
        d.loc[te.index, "score_"+v] = -m.score_samples(te[f])
print("fitted in", round(time.time()-t0), "s")
flags = pd.read_csv(OUT_DIR/"flags.csv.gz", parse_dates=["timestamp"])
add = d[["building_id","timestamp"]+["score_"+v for v in VARIANTS]]
fx = flags.merge(add, on=["building_id","timestamp"], how="left")
# validity: same hours as other methods AND iforest features present
cv = fx[(fx["split"]=="val") & fx["valid"] & (fx["label"]==0)]
cut = {}
for v in VARIANTS:
    fx.loc[~fx["valid"], "score_"+v] = np.nan
    cut[v] = float(np.nanquantile(cv["score_"+v].dropna(), 0.995))
    fx["flag_"+v] = fx["valid"] & (fx["score_"+v] > cut[v])
print("cutoffs", cut)
print(fx.groupby("split")[["flag_"+v for v in VARIANTS]].mean().round(4))
print("share valid hours without iforest score:", {v: float((fx["valid"] & fx["score_"+v].isna()).mean()) for v in VARIANTS})
fx.to_csv(OUT_DIR/"flags_ext.csv.gz", index=False)
json.dump({"cutoffs_iforest": cut, "n_estimators":100, "max_samples":256, "seed":0, "fit_on":"training rows before 2016-09-13, per building, no labels"}, open(OUT_DIR/"cutoffs_iforest.json","w"), indent=2)


## Part B. Equal alarm rates with 95% intervals grouped by building, delay and hour share (test)

In [ ]:
import sys, json
from config import *
FLAG = OUT_DIR/"flags_ext.csv.gz"
TAG = "ext"
RATES=[0.5,1,2]; MODES=["pooled","per_building"]; NB=2000
PRETTY={"last_hour":"Same as last hour","typical":"Historical typical value","A":"Model A","B":"Model B","C":"Model C","fixed":"Fixed percentile","iforest_C":"Isolation Forest (on model C error)","iforest_raw":"Isolation Forest (on raw readings)"}
f = pd.read_csv(FLAG, parse_dates=["timestamp"])
f = f[f["split"]=="test"].sort_values(["building_id","timestamp"]).reset_index(drop=True)
f["t"]=(f["timestamp"]-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
thr=pd.read_csv(OUT_DIR/"cut_fixed_thresholds.csv",index_col="building_id")["fixed_threshold"]
f["score_fixed"]=f["actual"]/f["building_id"].map(thr)
methods=[c[6:] for c in f.columns if c.startswith("score_")]
ev=pd.read_csv(OUT_DIR/"lead_events.csv",parse_dates=["start","end"])
ev["split"]=ev["split"].replace({"validation":"val"})
ev=ev[(ev["split"]=="test")&(ev["zero_share"]<=0.5)&ev["building_id"].isin(f["building_id"].unique())].copy()
ev["s"]=(ev["start"]-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
ev["e"]=(ev["end"]-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
bt={b:g["t"].to_numpy() for b,g in f[f["valid"]].groupby("building_id")}
cnt=lambda T,b,lo,hi:0 if T.get(b) is None else int(np.searchsorted(T[b],hi,"right")-np.searchsorted(T[b],lo,"left"))
ev["n_valid"]=[cnt(bt,b,s,e) for b,s,e in zip(ev["building_id"],ev["s"],ev["e"])]
ev=ev[ev["n_valid"]>0].reset_index(drop=True)
print("usable test events",len(ev),"methods",methods)
def mask(score,mode,r):
    s=pd.Series(score,index=f.index).where(f["valid"])
    if mode=="pooled": return (s>np.nanquantile(s.to_numpy(),1-r/100)).fillna(False).to_numpy()
    p=s.groupby(f["building_id"]).rank(pct=True,method="first")
    return (p>1-r/100).fillna(False).to_numpy()
def per_event(m):
    T={b:g["t"].to_numpy() for b,g in f[m].groupby("building_id")}
    found,delay,nf=[],[],[]
    for b,s,e in zip(ev["building_id"],ev["s"],ev["e"]):
        t=T.get(b)
        if t is None: found.append(False);delay.append(np.nan);nf.append(0);continue
        lo=np.searchsorted(t,s,"left");hi=np.searchsorted(t,e,"right")
        found.append(hi>lo);delay.append(float(t[lo]-s) if hi>lo else np.nan);nf.append(hi-lo)
    return np.array(found),np.array(delay),np.array(nf)
rng=np.random.default_rng(0)
bl=ev["building_id"].to_numpy(); ub=np.unique(bl)
idx_by_b={b:np.where(bl==b)[0] for b in ub}
boots=[np.concatenate([idx_by_b[b] for b in rng.choice(ub,len(ub))]) for _ in range(NB)]
rows=[];diff=[]
for mode in MODES:
  for r in RATES:
    store={}
    for m in methods:
        mk=mask(f["score_"+m].to_numpy(),mode,r)
        fo,de,nf=per_event(pd.Series(mk,index=f.index))
        store[m]=fo
        rec=fo.mean()*100
        bs=np.array([fo[i].mean()*100 for i in boots])
        inside=(nf/ev["n_valid"].to_numpy())
        rows.append(dict(mode=mode,rate_pct=r,method=PRETTY.get(m,m),events=len(ev),found=int(fo.sum()),recall_pct=round(rec,1),
            ci_low=round(np.percentile(bs,2.5),1),ci_high=round(np.percentile(bs,97.5),1),
            median_delay_h=round(float(np.nanmedian(de)),1) if fo.any() else np.nan,
            hour_share_pct=round(inside.mean()*100,1),
            hour_share_when_found_pct=round(inside[fo].mean()*100,1) if fo.any() else np.nan,
            false_alarm_pct=round(f.loc[mk&f["valid"]&(f["label"]==0),"valid"].sum()/max(1,(f["valid"]&(f["label"]==0)).sum())*100,3)))
    for a,b in [("C","last_hour"),("iforest_C","last_hour"),("iforest_raw","last_hour"),("iforest_C","C"),("C","typical")]:
        if a in store and b in store:
            d0=(store[a].mean()-store[b].mean())*100
            bs=np.array([(store[a][i].mean()-store[b][i].mean())*100 for i in boots])
            diff.append(dict(mode=mode,rate_pct=r,comparison=f"{PRETTY[a]} minus {PRETTY[b]}",diff_pp=round(d0,1),ci_low=round(np.percentile(bs,2.5),1),ci_high=round(np.percentile(bs,97.5),1)))
R=pd.DataFrame(rows);D=pd.DataFrame(diff)
R.to_csv(OUT_DIR/f"eval_equal_rate_ci_{TAG}.csv",index=False);D.to_csv(OUT_DIR/f"eval_equal_rate_diff_{TAG}.csv",index=False)
pd.set_option("display.width",250);pd.set_option("display.max_columns",30)
print(R[R.rate_pct==1].to_string(index=False));print(D.to_string(index=False))


## Part C. Check: per-building cut-offs taken from validation, applied to test

In [ ]:
import sys
from config import *
PRETTY={"last_hour":"Same as last hour","typical":"Historical typical value","A":"Model A","B":"Model B","C":"Model C","iforest_C":"Isolation Forest (model C error)","iforest_raw":"Isolation Forest (raw readings)"}
f=pd.read_csv(OUT_DIR/"flags_ext.csv.gz",parse_dates=["timestamp"])
f["t"]=(f["timestamp"]-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
ev=pd.read_csv(OUT_DIR/"lead_events.csv",parse_dates=["start","end"]);ev["split"]=ev["split"].replace({"validation":"val"})
ft=f[f["split"]=="test"]
ev=ev[(ev.split=="test")&(ev.zero_share<=0.5)&ev.building_id.isin(ft.building_id.unique())].copy()
ev["s"]=(ev.start-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1);ev["e"]=(ev.end-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
vt={b:g.t.to_numpy() for b,g in ft[ft.valid].groupby("building_id")}
ev["nv"]=[0 if vt.get(b) is None else int(np.searchsorted(vt[b],e,"right")-np.searchsorted(vt[b],s,"left")) for b,s,e in zip(ev.building_id,ev.s,ev.e)]
ev=ev[ev.nv>0].reset_index(drop=True)
cv=f[(f.split=="val")&f.valid&(f.label==0)]
rows=[]
for q in [0.5,1.0]:
  for m in PRETTY:
    sc="score_"+m
    pooled=np.nanquantile(cv[sc].dropna(),1-q/100)
    cnt=cv.groupby("building_id")[sc].count()
    pb=cv.groupby("building_id")[sc].quantile(1-q/100)
    cut=pb.where(cnt>=500)
    thr=ft.building_id.map(cut).fillna(pooled)
    for rule,th in [("one cut-off for all (current)",pooled),("own cut-off per building",thr)]:
        fl=ft.valid&(ft[sc]>th)
        T={b:g.t.to_numpy() for b,g in ft[fl].groupby("building_id")}
        fo=np.array([(lambda t:False if t is None else np.searchsorted(t,e,"right")>np.searchsorted(t,s,"left"))(T.get(b)) for b,s,e in zip(ev.building_id,ev.s,ev.e)])
        rows.append(dict(target_pct=q,rule=rule,method=PRETTY[m],test_flag_rate_pct=round(fl.sum()/ft.valid.sum()*100,3),events_found=int(fo.sum()),recall_pct=round(fo.mean()*100,1)))
R=pd.DataFrame(rows);R.to_csv(OUT_DIR/"eval_frozen_rules_test.csv",index=False)
pd.set_option("display.width",200);print(len(ev));print(R.to_string(index=False))
